# 02 — Modélisation v4 (GEE v3 + INS ECAM 4)

**Objectif :** Entraîner LightGBM avec features satellitaires **et** indicateurs INS régionaux,
comparer les performances à v3, analyser l'importance des variables INS.

> Variables INS : jointure régionale (ECAM 4, 2014) — voir limites de leakage régional.


In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.run_notebook_02_pipeline import FEATURE_COLUMNS_BY_SET, run_pipeline

V4_PARQUET = PROJECT_ROOT / "data/processed/features/cluster_features_gee_ins_v4.parquet"
FINAL_PARQUET = PROJECT_ROOT / "data/processed/final_features_v4.parquet"
REPORT = PROJECT_ROOT / "outputs/reports/model_v4_results.json"

print("✓ Setup v4")


✓ Setup v4


In [2]:
real_v4 = run_pipeline(
    feature_set="v4",
    use_fake=False,
    gee_parquet=V4_PARQUET,
    save_artifacts=True,
)
m = real_v4["metrics_oof"]
print(f"R² OOF       : {m['r2']:.4f}")
print(f"Spearman OOF : {m['spearman']:.4f}")
print(f"RMSE OOF     : {m['rmse']:.0f}")
print(f"MAE OOF      : {m['mae']:.0f}")


R² OOF       : 0.7934
Spearman OOF : 0.8886
RMSE OOF     : 38323
MAE OOF      : 29504


In [3]:
if REPORT.exists():
    report = json.loads(REPORT.read_text(encoding="utf-8"))
    comp = report["comparison_v3_vs_v4"]["metrics"]
    display(pd.DataFrame(comp).T)
    ins_imp = pd.DataFrame(report["ins_feature_importance"])
    display(ins_imp)


,r2,rmse,mae,spearman
v3,0.786710,38940.858549,30398.795552,0.881812
v4,0.793425,38322.983872,29504.109708,0.888552
delta_v4_minus_v3,0.006715,-617.874677,-894.685844,0.006740


,rank,feature,gain
0,1,ins_poverty_rate_pct,1.753823e+10
1,2,ins_literacy_rate_15plus_pct,3.606739e+11
2,3,ins_electricity_access_pct,0.000000e+00
3,4,ins_primary_enrollment_pct,2.062571e+10
